# Group 03 — Week 3: Augmented Training Across 3 YOLO Versions
**Objective:** Train each of the 3 YOLO versions (YOLOv8n, YOLOv11n, YOLOv26n) separately on every augmented dataset version (Low / Medium / High scaling-zoom intensity), keeping dataset split, image size, epochs, batch size and hardware identical to the corresponding Week 1 baseline run.

**This week produces 9 training runs** (3 intensities × 3 YOLO versions) and saves a trained weight file for each. Full metrics consolidation into a comparison matrix happens next week (Week 4) — this notebook focuses on generating the augmented datasets and getting all 9 models trained and saved.


## 1. Setup

In [1]:
!pip install -q ultralytics opencv-python-headless
from ultralytics import YOLO
import os, glob, shutil, time, random
import cv2
import numpy as np
import pandas as pd


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 765.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 3.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## 2. Reload Week 1's baseline dataset split (from Google Drive)
Mounts your Google Drive and extracts `baseline_dataset.zip` directly from there into Colab — faster and more reliable than the browser upload widget.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import glob, zipfile

# Locate the zip in Drive (adjust the search if you placed it in a subfolder)
zip_candidates = glob.glob('/content/drive/MyDrive/**/baseline_dataset.zip', recursive=True)
if not zip_candidates:
    raise FileNotFoundError("baseline_dataset.zip not found in Drive. Check the filename/location and update the search pattern above.")

zip_path = zip_candidates[0]
print(f"Found zip at: {zip_path}")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/content')

# Handle the common double-nested extraction path automatically
dataset_candidates = glob.glob('/content/**/dataset_split', recursive=True)
BASELINE_ROOT = dataset_candidates[0] if dataset_candidates else '/content/dataset_split'
print(f"Baseline dataset found at: {BASELINE_ROOT}")

CLASS_NAMES = ['car', 'threewheel', 'bus', 'truck', 'motorbike', 'van']

n_train = len(glob.glob(f'{BASELINE_ROOT}/train/images/*'))
n_val   = len(glob.glob(f'{BASELINE_ROOT}/val/images/*'))
n_test  = len(glob.glob(f'{BASELINE_ROOT}/test/images/*'))
print(f"train={n_train}  val={n_val}  test={n_test}")


Found zip at: /content/drive/MyDrive/baseline_dataset.zip
Baseline dataset found at: /content/content/dataset_split
train=2100  val=450  test=450


## 3. Re-declare the scaling/zoom augmentation (from Week 2)
Same function and intensity ranges as Week 2 — reused exactly as implemented and validated last week.


In [4]:
def load_yolo_labels(label_path):
    boxes = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls, xc, yc, w, h = parts[:5]
                boxes.append((int(cls), float(xc), float(yc), float(w), float(h)))
    return boxes

def save_yolo_labels(label_path, boxes):
    with open(label_path, 'w') as f:
        for cls, xc, yc, w, h in boxes:
            f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def scale_zoom_augment(img, boxes, scale, pad_color=(114, 114, 114)):
    h, w = img.shape[:2]
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    if scale >= 1:
        x0 = (new_w - w) // 2
        y0 = (new_h - h) // 2
        out_img = resized[y0:y0 + h, x0:x0 + w]
    else:
        out_img = np.full((h, w, 3), pad_color, dtype=np.uint8)
        x0 = (w - new_w) // 2
        y0 = (h - new_h) // 2
        out_img[y0:y0 + new_h, x0:x0 + new_w] = resized

    new_boxes = []
    for cls, xc, yc, bw, bh in boxes:
        xc2 = 0.5 + (xc - 0.5) * scale
        yc2 = 0.5 + (yc - 0.5) * scale
        bw2 = bw * scale
        bh2 = bh * scale

        x1, y1 = xc2 - bw2 / 2, yc2 - bh2 / 2
        x2, y2 = xc2 + bw2 / 2, yc2 + bh2 / 2
        x1c, y1c = max(0.0, x1), max(0.0, y1)
        x2c, y2c = min(1.0, x2), min(1.0, y2)

        if x2c <= x1c or y2c <= y1c:
            continue
        new_boxes.append((cls, (x1c + x2c) / 2, (y1c + y2c) / 2, x2c - x1c, y2c - y1c))

    return out_img, new_boxes

INTENSITY_LEVELS = {
    'Low':    (0.90, 1.10),
    'Medium': (0.75, 1.25),
    'High':   (0.60, 1.40),
}
print("Augmentation function and intensity levels restored.")


Augmentation function and intensity levels restored.


## 4. Build the 3 full augmented datasets
For each intensity level: every **training** image gets a fresh random scale factor drawn from that intensity's range (so the model sees varied zoom levels across the dataset, not one fixed value). **Val and test splits are copied unchanged** from the baseline — they must stay identical across all runs so evaluation is a fair comparison, per the project's "identical to the corresponding baseline run" requirement.

This only augments the train split; the number of training images stays the same (each original image's augmented version replaces it in this dataset copy, rather than being added alongside it) — so dataset size and training time stay comparable to baseline.


In [5]:
random.seed(42)  # reproducible scale factors per intensity

AUGMENTED_ROOT = '/content/augmented_datasets'

for intensity, (lo, hi) in INTENSITY_LEVELS.items():
    out_root = f'{AUGMENTED_ROOT}/{intensity}'
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{out_root}/{split}/images', exist_ok=True)
        os.makedirs(f'{out_root}/{split}/labels', exist_ok=True)

    # Augment TRAIN split only
    train_imgs = glob.glob(f'{BASELINE_ROOT}/train/images/*')
    for img_path in train_imgs:
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        label_path = f'{BASELINE_ROOT}/train/labels/{base}.txt'

        img = cv2.imread(img_path)
        boxes = load_yolo_labels(label_path)

        scale = random.uniform(lo, hi)
        aug_img, aug_boxes = scale_zoom_augment(img, boxes, scale)

        cv2.imwrite(f'{out_root}/train/images/{base}{ext}', aug_img)
        save_yolo_labels(f'{out_root}/train/labels/{base}.txt', aug_boxes)

    # Copy VAL and TEST unchanged (identical across baseline + all intensities)
    for split in ['val', 'test']:
        for img_path in glob.glob(f'{BASELINE_ROOT}/{split}/images/*'):
            base = os.path.splitext(os.path.basename(img_path))[0]
            ext = os.path.splitext(img_path)[1]
            shutil.copy(img_path, f'{out_root}/{split}/images/{base}{ext}')
            shutil.copy(f'{BASELINE_ROOT}/{split}/labels/{base}.txt', f'{out_root}/{split}/labels/{base}.txt')

    # data.yaml for this intensity
    yaml_content = f"""path: {out_root}
train: train/images
val: val/images
test: test/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
    with open(f'{out_root}/data.yaml', 'w') as f:
        f.write(yaml_content)

    n_train_aug = len(glob.glob(f'{out_root}/train/images/*'))
    print(f"{intensity}: train={n_train_aug} images augmented, val/test copied unchanged, data.yaml written")


Low: train=2100 images augmented, val/test copied unchanged, data.yaml written
Medium: train=2100 images augmented, val/test copied unchanged, data.yaml written
High: train=2100 images augmented, val/test copied unchanged, data.yaml written


## 5. Train all 9 models (3 intensities × 3 YOLO versions)
Same training config as the Week 1 baseline: 30 epochs, imgsz 640, batch 16, with Ultralytics' own built-in augmentations (mosaic, flips, HSV, etc.) explicitly disabled — so the **only** augmentation present is the scaling/zoom we applied to the dataset ourselves.

**Split into 9 separate cells below** (one per intensity × YOLO version) instead of one big loop — run them one at a time. This way, if a session drops or a cell errors out partway, you only need to rerun that one combination, not start the whole thing over. Each cell takes roughly 15–25 minutes.


In [6]:
EPOCHS = 30     # must match Week 1 baseline exactly
IMGSZ = 640
BATCH = 16

NO_AUG = dict(
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0,
    erasing=0.0,
)

MODEL_WEIGHTS = {
    'YOLOv8n':  'yolov8n.pt',
    'YOLOv11n': 'yolo11n.pt',
    'YOLOv26n': 'yolo26n.pt',
}

trained_models = {}   # (intensity, version_name) -> YOLO model object
run_timings = {}      # (intensity, version_name) -> train_time_seconds

def train_one(intensity, version_name):
    """Train a single (intensity, YOLO version) combination and store the result."""
    key = (intensity, version_name)
    if key in trained_models:
        print(f"{intensity}_{version_name} already trained this session — skipping. "
              f"Delete it from trained_models to force a retrain.")
        return

    data_yaml = f'{AUGMENTED_ROOT}/{intensity}/data.yaml'
    run_name = f'{intensity}_{version_name}'
    print(f"{'='*60}\nTraining {version_name} on {intensity}-intensity augmented data\n{'='*60}")

    model = YOLO(MODEL_WEIGHTS[version_name])
    t0 = time.time()
    model.train(
        data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        name=run_name, project='/content/runs_week3', **NO_AUG
    )
    train_time = time.time() - t0

    trained_models[key] = model
    run_timings[key] = train_time
    print(f"{run_name} finished in {train_time/60:.1f} min")

print("Config and train_one() ready. Run the 9 cells below, one at a time.")


Config and train_one() ready. Run the 9 cells below, one at a time.


In [7]:
train_one('Low', 'YOLOv8n')


Training YOLOv8n on Low-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Low/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=L

In [8]:
train_one('Low', 'YOLOv11n')


Training YOLOv11n on Low-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Low/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=

In [9]:
train_one('Low', 'YOLOv26n')


Training YOLOv26n on Low-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Low/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=

In [10]:
train_one('Medium', 'YOLOv8n')


Training YOLOv8n on Medium-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Medium/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, 

In [11]:
train_one('Medium', 'YOLOv11n')


Training YOLOv11n on Medium-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Medium/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0,

In [12]:
train_one('Medium', 'YOLOv26n')


Training YOLOv26n on Medium-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/Medium/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0,

In [13]:
train_one('High', 'YOLOv8n')


Training YOLOv8n on High-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/High/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name

In [14]:
train_one('High', 'YOLOv11n')


Training YOLOv11n on High-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/High/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, nam

In [15]:
train_one('High', 'YOLOv26n')


Training YOLOv26n on High-intensity augmented data
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_datasets/High/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, nam

## 6. Quick validation check on the TEST split for every run
A lightweight sanity pass confirming all 9 models trained successfully and produce reasonable metrics — full comparison-matrix analysis (intensity × YOLO version, against baseline) happens in Week 4.


In [16]:
rows = []
for (intensity, version_name), model in trained_models.items():
    data_yaml = f'{AUGMENTED_ROOT}/{intensity}/data.yaml'
    t0 = time.time()
    metrics = model.val(data=data_yaml, split='test')
    inference_time = (time.time() - t0)

    rows.append({
        'Intensity': intensity,
        'YOLO Version': version_name,
        'Precision': round(float(metrics.box.mp), 4),
        'Recall': round(float(metrics.box.mr), 4),
        'mAP50': round(float(metrics.box.map50), 4),
        'mAP50-95': round(float(metrics.box.map), 4),
        'Train Time (min)': round(run_timings[(intensity, version_name)] / 60, 1),
        'Inference Time (s, full test set)': round(inference_time, 1),
    })

week3_results_df = pd.DataFrame(rows)
week3_results_df.to_csv('/content/week3_augmented_results.csv', index=False)
week3_results_df


Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 65.8±60.5 MB/s, size: 157.0 KB)
val: Scanning /content/augmented_datasets/Low/test/labels... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 481.0it/s 0.9s
val: New cache created: /content/augmented_datasets/Low/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.6it/s 6.3s
                   all        450        612      0.946      0.872      0.929      0.825
                   car         94        113      0.958      0.816      0.877      0.832
            threewheel         72        115       0.98      0.852      0.926      0.833
                   bus         74         82      0.987      0.961      0.982      0.923
                 truck         73        100      0.906  

,Intensity,YOLO Version,Precision,Recall,mAP50,mAP50-95,Train Time (min),"Inference Time (s, full test set)"
0,Low,YOLOv8n,0.9455,0.8716,0.9295,0.8254,17.2,10.5
1,Low,YOLOv11n,0.9187,0.8800,0.9217,0.8140,19.1,11.4
2,Low,YOLOv26n,0.9717,0.9103,0.9593,0.8694,19.8,11.4
3,Medium,YOLOv8n,0.9398,0.8642,0.9275,0.8281,17.0,11.8
4,Medium,YOLOv11n,0.9388,0.8806,0.9241,0.8245,17.7,9.8
5,Medium,YOLOv26n,0.9596,0.8973,0.9532,0.8614,20.0,11.2
6,High,YOLOv8n,0.9355,0.8636,0.9206,0.8189,17.0,12.1
7,High,YOLOv11n,0.9555,0.8785,0.9138,0.8084,18.0,10.7
8,High,YOLOv26n,0.9476,0.8980,0.9402,0.8606,20.3,9.6


## 7. Package Week 3 deliverables
Zips all 9 trained weight files (`best.pt` for each intensity × YOLO version combination) plus the results CSV, ready for submission and for Week 4's full consolidation.


In [17]:
# Collect just the best.pt weight files (full runs_week3 folder also has logs/plots if you want everything)
import os

weights_out_dir = '/content/week3_weights'
os.makedirs(weights_out_dir, exist_ok=True)

for (intensity, version_name) in trained_models:
    run_name = f'{intensity}_{version_name}'
    src = f'/content/runs_week3/{run_name}/weights/best.pt'
    dst = f'{weights_out_dir}/{run_name}_best.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)

print("Collected weight files:")
for f in os.listdir(weights_out_dir):
    print(" ", f)


Collected weight files:
  Low_YOLOv26n_best.pt
  Medium_YOLOv8n_best.pt
  High_YOLOv11n_best.pt
  Medium_YOLOv11n_best.pt
  Low_YOLOv11n_best.pt
  Low_YOLOv8n_best.pt
  High_YOLOv8n_best.pt
  High_YOLOv26n_best.pt
  Medium_YOLOv26n_best.pt


In [18]:
!zip -r /content/week3_deliverables.zip /content/week3_weights /content/week3_augmented_results.csv /content/runs_week3

from google.colab import files
files.download('/content/week3_deliverables.zip')


  adding: content/week3_weights/ (stored 0%)
  adding: content/week3_weights/Low_YOLOv26n_best.pt (deflated 11%)
  adding: content/week3_weights/Medium_YOLOv8n_best.pt (deflated 9%)
  adding: content/week3_weights/High_YOLOv11n_best.pt (deflated 10%)
  adding: content/week3_weights/Medium_YOLOv11n_best.pt (deflated 10%)
  adding: content/week3_weights/Low_YOLOv11n_best.pt (deflated 10%)
  adding: content/week3_weights/Low_YOLOv8n_best.pt (deflated 9%)
  adding: content/week3_weights/High_YOLOv8n_best.pt (deflated 9%)
  adding: content/week3_weights/High_YOLOv26n_best.pt (deflated 11%)
  adding: content/week3_weights/Medium_YOLOv26n_best.pt (deflated 11%)
  adding: content/week3_augmented_results.csv (deflated 49%)
  adding: content/runs_week3/ (stored 0%)
  adding: content/runs_week3/Low_YOLOv11n/ (stored 0%)
  adding: content/runs_week3/Low_YOLOv11n/results.csv (deflated 60%)
  adding: content/runs_week3/Low_YOLOv11n/BoxR_curve.png (deflated 11%)
  adding: content/runs_week3/Low_YOLOv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next week (Week 4) preview
Week 4 consolidates results for Baseline vs. each chosen intensity level, for every YOLO version, into a single comparison matrix (intensity × YOLO version) — Precision, Recall, mAP@50, mAP@50-95, training time, and inference time for each of the 12 runs (1 baseline + 9 augmented, per version... actually 3 baseline + 9 augmented = 12 total). You already have the Week 1 baseline CSV and this week's `week3_augmented_results.csv` — next week is primarily about merging and analysing these two tables together.
